# ATML PA2 — Task 1: DPO

Use an **A100 GPU** runtime. This notebook launches repository scripts for all five experiments. Results and adapters are saved to Google Drive. It supplies numeric outputs and review records; write your report analysis yourself. Run cells in order, including the real-model smoke test.

**The default source is GitHub.** Open this notebook in Colab and run the cells in order. The optional `CODE_SOURCE = "zip"` mode supports an offline code archive.


## 1. Mount Drive and load the implementation

Keep the same `DRIVE_ROOT` on later sessions. Existing runs are resumed only when their code/data/settings match. Downloaded course assets remain on local Colab disk.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, sys, subprocess, zipfile
from pathlib import Path

CODE_SOURCE = 'github'  # Optional: 'zip' to upload a code archive.
DRIVE_ROOT = Path('/content/drive/MyDrive/PA2')
REPOSITORY = 'https://github.com/WasifAhmed1917/ATML.git'
REPO_REF = 'main'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if CODE_SOURCE == 'zip':
    CHECKOUT = Path('/content/ATML_PA2_package')
    print('Upload ATML_PA2_Task1.zip supplied with this notebook.')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, 'Upload exactly one PA2 package zip'
    CHECKOUT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            destination = (CHECKOUT / item.filename).resolve()
            assert destination.is_relative_to(CHECKOUT.resolve()), 'Unsafe archive path'
            assert item.filename.startswith('PA2/'), 'Expected a PA2/ package'
        archive.extractall(CHECKOUT)
elif CODE_SOURCE == 'github':
    CHECKOUT = Path('/content/ATML')
    if not CHECKOUT.exists():
        subprocess.run(['git', 'clone', '--branch', REPO_REF, REPOSITORY, str(CHECKOUT)], check=True)
else:
    raise ValueError('CODE_SOURCE must be zip or github')
PA2 = CHECKOUT / 'PA2'
assert (PA2 / 'task1_dpo/run_all.py').exists(), 'Task 1 implementation is missing'
os.chdir(PA2)
os.environ['PA2_ARTIFACT_ROOT'] = str(DRIVE_ROOT)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
revision_file = PA2 / 'SOURCE_REVISION.txt'
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() if CODE_SOURCE == 'github' else revision_file.read_text().strip() if revision_file.exists() else 'unknown archive revision'
print('Code revision:', revision)
print('Persistent artifacts:', DRIVE_ROOT)


## 2. Install the released environment

This runs in a separate process. Do not change the supplied training constants to fit a smaller GPU.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

def run(module, *args, artifact_root=None):
    env = os.environ.copy()
    if artifact_root is not None:
        env['PA2_ARTIFACT_ROOT'] = str(artifact_root)
    command = [sys.executable, '-m', module, *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=PA2, env=env, check=True)

subprocess.run(['nvidia-smi'], check=True)
run('scripts.check_environment')


## 3. Download and validate the pinned course assets

The course downloader installs fixed files, caches, and supplied checkpoints. The Hugging Face assets repository does not need a manual clone.

In [ ]:
run('scripts.download_assets')
run('scripts.validate_assets')
run('task1_dpo.validate_data', '--tokenize')


## 4. Validate the objective and pipeline on CPU fixtures

These are software checks, not experiment results.

In [ ]:
run('unittest', 'discover', '-s', 'tests', '-v')


## 5. Real-model GPU smoke test

Checks actual Qwen loading, one update, generation, KL, and frozen reward scoring. This uses a separate local artifact directory and is never exported as a full course run. If this fails, resolve the failure before running full experiments.

In [ ]:
SMOKE_ROOT = Path('/content/PA2_task1_smoke')
run('task1_dpo.train', '--run-name', 'smoke', '--output', 'outputs/task1_dpo/smoke',
    '--smoke', '--resume', '--skip-completed', artifact_root=SMOKE_ROOT)
run('task1_dpo.evaluate', '--adapter', 'outputs/task1_dpo/smoke', '--name', 'smoke',
    '--length', '--smoke', '--resume', artifact_root=SMOKE_ROOT)


## 6. Standard DPO and optional SFT baseline

Standard: 1,500 fixed pairs, one epoch, beta .10. This also evaluates the stratified and word-limit sets. Saves the standard adapter needed for Task 4. SFT adds generation evidence for manual comparison.

In [ ]:
run('task1_dpo.run_all', '--stage', 'standard', '--resume')


## 7. Three matched beta forks

Beta .03, .10, .30; each starts from the original Qwen initialization and uses the same first 600 pairs. These do not continue the standard adapter.

In [ ]:
run('task1_dpo.run_all', '--stage', 'beta', '--resume')


## 8. Length-balanced DPO

Train on the fixed 1,500-pair balanced file without resampling. Compare with the saved standard run on identical held-out strata, prompts, and word-limit prompts.

In [ ]:
run('task1_dpo.run_all', '--stage', 'length', '--resume')


## 9. Export, check completeness, and plot saved results

No model inference is required for exports and plots. These files can be reloaded and replotted in a later session.

In [ ]:
run('task1_dpo.run_all', '--stage', 'export', '--resume')
run('task1_dpo.plot_results')


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
RESULTS = DRIVE_ROOT / 'results/task1_dpo'
for filename in ['summary.csv', 'length_strata.csv', 'word_limit_summary.csv']:
    print(filename)
    display(pd.read_csv(RESULTS / filename))
print(json.loads((RESULTS / 'completion_check.json').read_text()))
for filename in ['beta_metrics.png', 'length_strata.png', 'word_limits.png', 'training_loss.png']:
    display(Image(filename=str(RESULTS / 'figures' / filename)))


## 10. Inspect evidence and package small results

Inspect `preprocessing_diagnostics.csv` and `qualitative_review.jsonl` yourself. Select and interpret failure cases for the report. Adapters are under `PA2/outputs/task1_dpo/` in Drive; results are under `PA2/results/task1_dpo/`. The following optional cell bundles results for GitHub submission, excluding adapters and public weights.

In [ ]:
import shutil
archive = shutil.make_archive('/content/ATML_PA2_T1_results', 'zip', root_dir=DRIVE_ROOT / 'results', base_dir='task1_dpo')
print('Results archive:', archive)
# Optional download:
# from google.colab import files
# files.download(archive)


## Resume in a new session

Rerun setup, dependency installation, and asset validation, then rerun the unfinished stage with `--resume` (already included). Matching completed training is skipped; evaluation resumes from saved records. A code/config/data mismatch intentionally stops rather than combining incompatible results. For plotting only, mount Drive and load the saved CSVs without downloading models or rerunning training.